# 데이터 전처리

In [56]:
import numpy as np,gc
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
plt.rc('font',family='Malgun Gothic' )
!pip install koreanize-matplotlib
from IPython.display import display
import koreanize_matplotlib
import seaborn as sns
import scipy as sp
from scipy import stats
import datetime
import os
import seaborn.objects as so
import statsmodels.api as sm

In [57]:
# 데이터 불러오기
train_transaction = pd.read_csv('data/train_transaction.csv')
train_identity = pd.read_csv('data/train_identity.csv')
test_transaction = pd.read_csv('data/test_transaction.csv')
test_identity = pd.read_csv('data/test_identity.csv')

In [58]:
# transaction과 identity를 TransactionID를 기준으로 병합
train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
test = pd.merge(test_transaction, test_identity, on='TransactionID', how='left')

In [59]:
# ID 컬럼 제거
train.drop(columns=['TransactionID'], inplace=True)

In [60]:
# test id-1 -> id_1
test.columns = test.columns.str.replace('-', '_')

In [61]:
# 1. card1 기준으로 연결된 card2가 유일한 경우만 찾기
card1_card2_map = train.groupby('card1')['card2'].nunique()

# 2. 유일한 card2 값을 가진 card1만 선택
card1_unique = card1_card2_map[card1_card2_map == 1].index

# 3. 해당 card1의 card2 값을 딕셔너리로 추출
card1_to_card2 = train[train['card1'].isin(card1_unique)].groupby('card1')['card2'].first().to_dict()

# 4. train 결측치 채우기
train.loc[train['card2'].isna() & train['card1'].isin(card1_to_card2), 'card2'] = \
    train.loc[train['card2'].isna() & train['card1'].isin(card1_to_card2), 'card1'].map(card1_to_card2)

# 1. card1 기준으로 연결된 card2가 유일한 경우만 찾기
card1_card2_map = test.groupby('card1')['card2'].nunique()

# 2. 유일한 card2 값을 가진 card1만 선택
card1_unique = card1_card2_map[card1_card2_map == 1].index

# 3. 해당 card1의 card2 값을 딕셔너리로 추출
card1_to_card2 = test[test['card1'].isin(card1_unique)].groupby('card1')['card2'].first().to_dict()

# 4. test 결측치 채우기
test.loc[test['card2'].isna() & test['card1'].isin(card1_to_card2), 'card2'] = \
    test.loc[test['card2'].isna() & test['card1'].isin(card1_to_card2), 'card1'].map(card1_to_card2)

In [62]:
# 분포를 기반으로 결측치를 채우기기
def fill_missing_by_sampling(df, col):
    # 확률 분포 계산
    value_dist = df[col].value_counts(normalize=True)
    
    # 결측치 인덱스
    na_indices = df[df[col].isna()].index
    
    # 샘플링하여 결측치 채움
    if not value_dist.empty:
        df.loc[na_indices, col] = np.random.choice(
            value_dist.index, size=len(na_indices), p=value_dist.values
        )
    return df

cols_to_fill = ['card2', 'card3', 'card4', 'card5', 'card6']

for col in cols_to_fill:
    train = fill_missing_by_sampling(train, col)
    test = fill_missing_by_sampling(test, col)

In [63]:
# 최빈값으로 결측치 채우기
def fill_missing_with_mode(df, col):
    if df[col].isnull().any():
        mode = df[col].mode()
        if not mode.empty:
            df[col].fillna(mode[0], inplace=True)
    return df

cols_to_fill = ['addr2']

for col in cols_to_fill:
    train = fill_missing_with_mode(train, col)
    test = fill_missing_with_mode(test, col)

C:\Users\chldu\AppData\Local\Temp\ipykernel_1848\1927018675.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.




In [64]:
# 결측치 비율이 0.9 이상인 컬럼 제거
train.drop(columns=['dist2'], inplace=True)
test.drop(columns=['dist2'], inplace=True)

In [65]:
# 이메일 도메인 그룹화
email_groups = {
    "gmail.com": "Gmail", "gmail": "Gmail",
    "yahoo.com": "Yahoo", "ymail.com": "Yahoo", "rocketmail.com": "Yahoo", 
    "yahoo.com.mx": "Yahoo", "yahoo.fr": "Yahoo", "yahoo.es": "Yahoo", 
    "yahoo.de": "Yahoo", "yahoo.co.uk": "Yahoo", "yahoo.co.jp": "Yahoo", 
    "hotmail.com": "Microsoft", "outlook.com": "Microsoft", "msn.com": "Microsoft", 
    "live.com": "Microsoft", "live.com.mx": "Microsoft", "outlook.es": "Microsoft", 
    "hotmail.es": "Microsoft", "hotmail.fr": "Microsoft", "hotmail.co.uk": "Microsoft", 
    "hotmail.de": "Microsoft", "live.fr": "Microsoft",
    "aol.com": "AOL", "aim.com": "AOL", 
    "comcast.net": "Comcast", 
    "icloud.com": "Apple", "me.com": "Apple", "mac.com": "Apple", 
    "sbcglobal.net": "AT&T", "att.net": "AT&T", "bellsouth.net": "AT&T", "prodigy.net.mx": "AT&T", 
    "verizon.net": "Verizon", "frontier.com": "Verizon", "frontiernet.net": "Verizon", 
    "cox.net": "Cox", 
    "charter.net": "Charter", 
    "optonline.net": "Optimum", 
    "earthlink.net": "EarthLink", 
    "windstream.net": "Windstream", 
    "embarqmail.com": "CenturyLink", "centurylink.net": "CenturyLink", "q.com": "CenturyLink", 
    "suddenlink.net": "Suddenlink", 
    "netzero.com": "NetZero", "netzero.net": "NetZero", 
    "twc.com": "TWC", "roadrunner.com": "TWC", "sc.rr.com": "TWC", "cfl.rr.com": "TWC", 
    "gmx.de": "GMX", "web.de": "GMX", 
    "mail.com": "Mail.com", 
    "protonmail.com": "ProtonMail", 
    "servicios-ta.com": "Servicios-TA", 
    "scranton.edu": "Scranton",
    "cableone.net" : "Cableone",
    "anonymous.com" : "Anonymous"
}

tmp1 = train["P_emaildomain"].apply(lambda x : email_groups.get(x, x))
tmp2 = train["R_emaildomain"].apply(lambda x : email_groups.get(x, x))

train["P_emaildomain"] = tmp1
train["R_emaildomain"] = tmp2

tmp3 = test["P_emaildomain"].apply(lambda x : email_groups.get(x, x))
tmp4 = test["R_emaildomain"].apply(lambda x : email_groups.get(x, x))

test["P_emaildomain"] = tmp3
test["R_emaildomain"] = tmp4

In [66]:
# 전체 V열 목록
V_cols = [f'V{i}' for i in range(1, 340)]

# 남길 V열 인덱스
V_tmp = []
V_tmp += [1, 3, 4, 6, 8, 11]
V_tmp += [13, 14, 17, 20, 23, 26, 27, 30]
V_tmp += [36, 37, 40, 41, 44, 47, 48]
V_tmp += [54, 56, 59, 62, 65, 67, 68, 70]
V_tmp += [76, 78, 80, 82, 86, 88, 89, 91]
V_tmp += [96, 98, 99, 104]
V_tmp += [107, 108, 111, 115, 117, 120, 121, 123]
V_tmp += [124, 127, 129, 130, 136]
V_tmp += [138, 139, 142, 147, 156, 162]
V_tmp += [165, 160, 166]
V_tmp += [178, 176, 173, 182]
V_tmp += [187, 203, 205, 207, 215]
V_tmp += [169, 171, 175, 180, 185, 188, 198, 210, 209]
V_tmp += [218, 223, 224, 226, 228, 229, 235]
V_tmp += [240, 258, 257, 253, 252, 260, 261]
V_tmp += [264, 266, 267, 274, 277]
V_tmp += [220, 221, 234, 238, 250, 271]
V_tmp += [294, 284, 285, 286, 291, 297]
V_tmp += [303, 305, 307, 309, 310, 320]
V_tmp += [281, 283, 289, 296, 301, 314]
V_tmp += [332, 325, 335, 338]

# 남길 V열 이름
V_selected = [f'V{i}' for i in V_tmp]

# 제거할 V열 (train 기준)
V_drop = [col for col in V_cols if col not in V_selected and col in train.columns]

# drop
train = train.drop(columns=V_drop)
test = test.drop(columns=[col for col in V_drop if col in test.columns])

In [67]:
# 제거할 열 리스트 생성 (id-7, id-8, id-21 ~ id-27)
id_selected = [f'id_{i}' for i in range(7, 9)] + [f'id_{i}' for i in range(21, 28)]

# train과 test 데이터에서 해당 열 제거
train = train.drop(columns=id_selected, errors='ignore')
test = test.drop(columns=id_selected, errors='ignore')

In [68]:
# 남은 결측치 처리
for df in [train, test]:
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col].fillna(-999, inplace=True)

C:\Users\chldu\AppData\Local\Temp\ipykernel_1848\2808124128.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.




In [69]:
# 결측치를 "unknown"으로 채울 열 목록
cols_to_fill = ["P_emaildomain", "R_emaildomain",
                'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
                'id_12','id_15', 'id_16','id_28','id_29', 'id_30','id_31','id_32','id_33','id_34','id_35','id_36','id_37','id_38',
                'DeviceType', 'DeviceInfo']

# train과 test 데이터의 해당 열에서 결측치 채우기
train[cols_to_fill] = train[cols_to_fill].fillna("unknown")
test[cols_to_fill] = test[cols_to_fill].fillna("unknown")

## 파생변수

In [70]:
## D1, D10, D15 열을 이용하여 새로운 열 생성
# 첫 거래까지 걸린 날짜
train['D1n'] = np.floor(train.TransactionDT / (24*60*60)) - train.D1
train['D10n'] = np.floor(train.TransactionDT / (24*60*60)) - train.D10
train['D15n'] = np.floor(train.TransactionDT / (24*60*60)) - train.D15
test['D1n'] = np.floor(test.TransactionDT / (24*60*60)) - test.D1
test['D10n'] = np.floor(test.TransactionDT / (24*60*60)) - test.D10
test['D15n'] = np.floor(test.TransactionDT / (24*60*60)) - test.D15

In [71]:
# 필요한 열만 추출해서 UID 생성
# UID를 생성할 열 목록
cols_for_uid = ['card1', 'addr1','dist1','D1n','M6',"P_emaildomain","DeviceType","DeviceInfo"]

# 문자열로 변환한 뒤, '-'로 연결
train['uid'] = train[cols_for_uid].astype(str).agg('_'.join, axis=1)
test['uid'] = test[cols_for_uid].astype(str).agg('_'.join, axis=1)

In [72]:
# train UID별 count 계산
uid_counts = train['uid'].value_counts().rename('uid_count')

# UID별 평균 및 표준편차 계산 (예: 'TransactionAmt' 기준으로)
uid_stats = train.groupby('uid')['TransactionAmt'].agg(['mean', 'std']).rename(
    columns={'mean': 'uid_mean', 'std': 'uid_std'}
)

# df에 병합
train = train.merge(uid_counts, how='left', left_on='uid', right_index=True)
train = train.merge(uid_stats, how='left', left_on='uid', right_index=True)

# test UID별 count 계산
uid_counts = test['uid'].value_counts().rename('uid_count')

# UID별 평균 및 표준편차 계산 (예: 'TransactionAmt' 기준으로)
uid_stats = test.groupby('uid')['TransactionAmt'].agg(['mean', 'std']).rename(
    columns={'mean': 'uid_mean', 'std': 'uid_std'}
)

# df에 병합
test = test.merge(uid_counts, how='left', left_on='uid', right_index=True)
test = test.merge(uid_stats, how='left', left_on='uid', right_index=True)

In [73]:
# 날짜 변환
START_DATE = '2017-12-01'
startdate = datetime.datetime.strptime(START_DATE, '%Y-%m-%d')
date_train = train['TransactionDT'].apply(lambda x: (startdate + datetime.timedelta(seconds = x)))

train['Year'] = date_train.dt.year
train['Month'] = date_train.dt.month
train['Weekday'] = date_train.dt.dayofweek
train['Hour'] = date_train.dt.hour
train['Day'] = date_train.dt.day

train.drop(["Year", "Month"], axis=1, inplace=True)

startdate = datetime.datetime.strptime(START_DATE, '%Y-%m-%d')
date_test = test['TransactionDT'].apply(lambda x: (startdate + datetime.timedelta(seconds = x)))

test['Year'] = date_test.dt.year
test['Month'] = date_test.dt.month
test['Weekday'] = date_test.dt.dayofweek
test['Hour'] = date_test.dt.hour
test['Day'] = date_test.dt.day

test.drop(["Year", "Month"], axis=1, inplace=True)

In [74]:
# 실수형 열의 결측치를 -999로 채우기
train["uid_std"].fillna(-999, inplace=True)
test["uid_std"].fillna(-999, inplace=True)

C:\Users\chldu\AppData\Local\Temp\ipykernel_1848\21085844.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


C:\Users\chldu\AppData\Local\Temp\ipykernel_1848\21085844.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', t

In [75]:
## 가려지는 열 전부 출력 
#모든 열 출력
pd.set_option('display.max_columns', None)

#모든 행 출력 (너무 많으면 경고)
pd.set_option('display.max_rows', None)

#너무 긴 데이터도 줄바꿈 없이 전체 출력
pd.set_option('display.expand_frame_repr', False)

#각 열의 너비 제한 해제 (기본값: 50)
pd.set_option('display.max_colwidth', None)

sns.set_theme(style="dark",
              palette= "deep")

In [76]:
# 결측치 개수 확인
def resumetable(df):
    summary = pd.DataFrame(df.dtypes, columns=['dtypes'])
    summary['missing(%)'] = df.isnull().sum().values / len(df) * 100
    summary['#missing'] = df.isnull().sum().values
    summary['#unique'] = df.nunique().values
    summary['first_value'] = df.iloc[0].values
    summary['second_value'] = df.iloc[1].values
    summary['third_value'] = df.iloc[2].values
    return summary

resumetable(train)

,dtypes,missing(%),#missing,#unique,first_value,second_value,third_value
isFraud,int64,0.0,0,2,0,0,0
TransactionDT,int64,0.0,0,573349,86400,86401,86469
TransactionAmt,float64,0.0,0,20902,68.5,29.0,59.0
ProductCD,object,0.0,0,5,W,W,W
card1,int64,0.0,0,13553,13926,2755,4663
card2,float64,0.0,0,500,327.0,404.0,490.0
card3,float64,0.0,0,114,150.0,150.0,150.0
card4,object,0.0,0,4,discover,mastercard,visa
card5,float64,0.0,0,119,142.0,102.0,166.0
card6,object,0.0,0,4,credit,credit,debit


In [77]:
# 파일 저장
train.to_csv("train_preprocessed.csv", index=False)
test.to_csv("test_preprocessed.csv", index=False)